# One-Month-Ahead Inflation Forecast, China: XGBoost
Based on the course notebook **5_Recession_Model_XGBoost**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score #regression metrics
from xgboost import XGBRegressor #XGBClassifier -> XGBRegressor: inflation is a flat number hence this is a regresion

import warnings
warnings.filterwarnings('ignore')

## 1. Group preprocessing

In [2]:
df = pd.read_csv(
    "china_economic_raw.csv",
    sep=";",
    na_values=["#DIV/0!", "#N/A", "#VALUE!", "#REF!"]
)

df.columns = df.columns.str.strip()

# Convert Date column
df["Date"] = pd.to_datetime(df["Date"], format="%b %y")

# Sort chronologically
df = df.sort_values("Date").reset_index(drop=True)

# Define target and predictors
target = "INFL"
predictors = [
    col for col in df.columns
    if col.startswith("X") and col[1:].isdigit()
]

# Basic inspection
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

# Convert predictors and target to numeric values

for col in predictors + ["INFL"]:
    original = df[col]

    converted = pd.to_numeric(
        original.astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

    # Check if conversion introduced additional NaNs
    new_missing = original.notna() & converted.isna()

    if new_missing.any():
        print(f"{col}: Unexpected values:")
        print(original[new_missing].unique()[:10])

    df[col] = converted

# Check whether all variables are numeric
print(df[predictors].dtypes.value_counts())

# Create a fresh copy of the dataset
data = df.sort_values("Date").reset_index(drop=True).copy()

# Convert all predictors to numeric
for col in predictors:
    original = data[col]

    converted = pd.to_numeric(
        original.astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

    # Check whether conversion creates new missing values
    new_missing = original.notna() & converted.isna()

    if new_missing.any():
        raise ValueError(
            f"Non-numeric values found in {col}: "
            f"{original[new_missing].unique()[:10]}"
        )

    data[col] = converted.astype(float)

# Keep original numeric values for duplicate detection
original_values = data[predictors].copy()

# Forward fill short gaps
data[predictors] = data[predictors].ffill(limit=2)

# Detect and merge duplicate features
features = predictors.copy()
merged_pairs = []

for i, keep in enumerate(predictors):
    if keep not in features:
        continue

    for remove in predictors[i + 1:]:
        if remove not in features:
            continue

        a = original_values[keep]
        b = original_values[remove]

        # Compare only observations where both have values
        overlap = a.notna() & b.notna()

        if overlap.sum() < 24:
            continue

        if a[overlap].nunique() < 3:
            continue

        # Check whether values match
        if np.allclose(
            a[overlap].to_numpy(dtype=float),
            b[overlap].to_numpy(dtype=float),
            rtol=1e-9,
            atol=1e-8
        ):

            # Combine available observations
            merged = a.combine_first(b)

            original_values[keep] = merged
            data[keep] = merged.ffill(limit=2)

            # Remove redundant column
            data = data.drop(columns=remove)
            features.remove(remove)

            merged_pairs.append((keep, remove))

print("Merged duplicate columns:", merged_pairs)
print("Remaining features:", len(features))

# Reference period for initial feature filtering
reference_mask = data["Date"].between(
    "2005-01-01", "2023-12-01"
)

# Missing values per feature
missing_ratio = (
    data.loc[reference_mask, features]
    .isna()
    .mean()
)

# Display highest missing ratios
print(
    (missing_ratio * 100)
    .sort_values(ascending=False)
    .head(20)
)

# Remove features with more than 20% missing
threshold = 0.20

drop_missing = missing_ratio[
    missing_ratio > threshold
].index.tolist()

# Identify constant features
drop_constant = [
    col for col in features
    if data.loc[
        reference_mask, col
    ].nunique(dropna=True) <= 1
]

# Combine exclusions
drop_features = list(set(
    drop_missing + drop_constant
))

# Remove features
data = data.drop(columns=drop_features)

features = [
    col for col in features
    if col not in drop_features
]

print("Dropped for missing values:", drop_missing)
print("Dropped constant features:", drop_constant)
print("Remaining features:", len(features))

FileNotFoundError: [Errno 2] No such file or directory: 'china_economic_raw.csv'

In [ ]:
# Shift all predictors by one month
X = data[features].shift(1).copy()

X.columns = [
    col + "_lag1" for col in features
]

# Historical inflation
X["INFL_lag1"] = data["INFL"].shift(1)
X["INFL_lag3"] = data["INFL"].shift(3)
X["INFL_lag12"] = data["INFL"].shift(12)

# Average inflation over the previous 3 months
X["INFL_mean3"] = (
    data["INFL"].shift(1).rolling(3).mean()
)

month = data["Date"].dt.month

X["month_sin"] = np.sin(2 * np.pi * month / 12)
X["month_cos"] = np.cos(2 * np.pi * month / 12)

# Combine target and engineered features
prepared = pd.concat(
    [data[["Date", "INFL"]], X],
    axis=1
)

# Keep only observations from 2005 onward
prepared = prepared[
    (prepared["Date"] >= "2005-01-01") &
    prepared["INFL"].notna()
].copy()

# Training observations only
outlier_train = prepared[
    prepared["Date"] < "2024-01-01"
].copy()

# Exclude binary and seasonal features from IQR audit
outlier_cols = [
    col for col in X.columns
    if col not in [
        "PMI_expansion_lag1",
        "month_sin",
        "month_cos"
    ]
]

Q1 = outlier_train[outlier_cols].quantile(0.25)
Q3 = outlier_train[outlier_cols].quantile(0.75)
IQR = Q3 - Q1

outliers = (
    (outlier_train[outlier_cols] < Q1 - 1.5 * IQR) |
    (outlier_train[outlier_cols] > Q3 + 1.5 * IQR)
)

print(outliers.sum().sort_values(ascending=False).head(20))

# Chronological train-test split

# Training: January 2005 - December 2023
train = prepared[
    (prepared["Date"] >= "2005-01-01") &
    (prepared["Date"] < "2024-01-01")
].copy()

# Testing: January 2024 - April 2026
test = prepared[
    (prepared["Date"] >= "2024-01-01") &
    (prepared["Date"] <= "2026-04-01")
].copy()

# Separate predictors and target
X_train = train.drop(columns=["Date", "INFL"])
y_train = train["INFL"]

X_test = test.drop(columns=["Date", "INFL"])
y_test = test["INFL"]

# Check number of observations
print("Training observations:", len(train))
print("Testing observations:", len(test))

## 2. PCA

In [ ]:
for n_comp in [1, 2, 3, 5, 10, 15, 20, 30, 50]:
    steps_X = [('imputer', SimpleImputer(strategy='median')),
               ('scaler', StandardScaler()),
               ('pca', PCA(n_components=n_comp, random_state=1))]
    X_pca_model = Pipeline(steps_X).fit(X_train)
    print('n_components:', n_comp, '  variance explained: %.3f' % X_pca_model.named_steps['pca'].explained_variance_ratio_.sum())

## 3. Pipeline
Course pipeline: scaler → PCA → XGBoost.
`XGBClassifier` becomes `XGBRegressor` with `objective='reg:squarederror'` (squared-error loss for a continuous target)

In [ ]:
# Course pipeline: imputer (for missing values) + scaler + PCA + XGBoost 
pipe_pca = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(random_state=1)),
    ('xgboost', XGBRegressor(objective='reg:squarederror', random_state=1))
])

# Same model without PCA (all features)
pipe_all = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('xgboost', XGBRegressor(objective='reg:squarederror', random_state=1))
])

## 4. Grid search for optimal hyperparameters 
The course test each hyperparameter combinattion with walk-forward validation over a validation P. Here we use `TimeSeriesSplit` (5 folds) inside the training period.

Hyperparameters
- `n_estimators`: number of trees
- `learning_rate`: how much each new tree corrects the previous ones
- `max_depth`: size of each tree
- `reg_alpha`: L1 penalty on leaf weights (XGBoost's regularization)
- `pca__n_components`: number of principal components (PCA pipeline only)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

grid_pca = {'pca__n_components':      [3, 5, 8, 10],
            'xgboost__n_estimators':  [100, 300],
            'xgboost__learning_rate': [0.05, 0.1, 0.3],
            'xgboost__max_depth':     [2, 3], #to stop the tree from overfitting
            'xgboost__reg_alpha':     [0, 0.5]}

grid_all = {k: v for k, v in grid_pca.items() if not k.startswith('pca')} #same grid withut PCA

search_pca = GridSearchCV(pipe_pca, grid_pca, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1).fit(X_train, y_train)
search_all = GridSearchCV(pipe_all, grid_all, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1).fit(X_train, y_train)

print('PCA + XGBoost best parameters:', search_pca.best_params_)
print('validation RMSE: %.4f' % -search_pca.best_score_)
print('XGBoost (all) best parameters:', search_all.best_params_)
print('validation RMSE: %.4f' % -search_all.best_score_)

## 5. Final model evaluation on the test period 
`GridSearchCV` automatically re-trains the best model on the full training period (2005–2023). We then forecast the 28 test months (2024–2026).
We report RMSE, MAE and R² and compare with the group's three benchmarks.

In [ ]:
def regression_metrics(actual, predicted):
    return {'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'MAE':  mean_absolute_error(actual, predicted),
            'R2':   r2_score(actual, predicted)}

benchmarks = {
    'Last-month inflation': X_test['INFL_lag1'].to_numpy(float),
    'Training mean': np.full(len(y_test), float(y_train.mean())),
    'Same month last year': X_test['INFL_lag12'].to_numpy(float),
}

In [ ]:
best_pca = search_pca.best_estimator_
best_all = search_all.best_estimator_
xgb_pca_pred = best_pca.predict(X_test)
xgb_all_pred = best_all.predict(X_test)

results = pd.DataFrame({'XGBoost (all features)': regression_metrics(y_test, xgb_all_pred),
                        'PCA + XGBoost': regression_metrics(y_test, xgb_pca_pred),
                        **{name: regression_metrics(y_test, p) for name, p in benchmarks.items()}}).T
results.round(4)

## 6. Forecast chart 

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test['Date'], y_test, label='Actual inflation', marker='o', color='black')
plt.plot(test['Date'], xgb_all_pred, label='XGBoost (all features)', marker='o')
plt.plot(test['Date'], xgb_pca_pred, label='PCA + XGBoost', marker='o', linestyle='--')
plt.title('China: One-Month-Ahead Inflation Forecast (XGBoost)')
plt.xlabel('Target month'); plt.ylabel('Monthly inflation (%)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('xgb_forecast.png', dpi=200); plt.show()

## 7. Feature importance
Which lagged indicators the XGBoost model (all features) relies on most. Importance = average improvement in the loss when the feature is used for a split (gain). These are predictive relationships, not causal ones.

In [ ]:
importance = pd.Series(best_all.named_steps['xgboost'].feature_importances_, index=X_train.columns)
top15 = importance.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
plt.barh(top15.index[::-1], top15.values[::-1])
plt.title('XGBoost: Top 15 Features'); plt.xlabel('Importance (gain)')
plt.tight_layout(); plt.savefig('xgb_feature_importance.png', dpi=200); plt.show()

In [ ]:
pd.DataFrame({'Date': test['Date'].values, 'Actual': y_test.values,
              'XGBoost_all': xgb_all_pred, 'XGBoost_PCA': xgb_pca_pred}).to_csv('xgb_predictions.csv', index=False)